# 2.0 - Machine Learning y LLMs

**MITSUI & CO. Commodity Prediction Challenge** (Kaggle) — versión simplificada.

Modelamos **un único target**:

> `target_4` = **LME_AH_Close - JPX_Gold_Standard_Futures_Close** (lag 1)

Benchmark de 3 algoritmos (Linear Regression, Decision Tree, HistGradientBoosting)
evaluados con **validación cruzada temporal** (`TimeSeriesSplit`): cada fold
entrena con el pasado y valida con el futuro inmediato.

Entrada: `X.parquet` / `y.parquet` de `data/processed/` (ver notebook 1.0).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.models.train_model import (
    MODEL_NAMES, benchmark, cross_validate_model, fit_full, save_model,
)
from src.models.predict_model import predict, save_predictions

config.TARGET, config.TARGET_DEFINITION, MODEL_NAMES

## Machine Learning — benchmark con CV temporal

In [ ]:
X = pd.read_parquet(config.PROCESSED_DATA_DIR / 'X.parquet')
y = pd.read_parquet(config.PROCESSED_DATA_DIR / 'y.parquet')
X.shape, y.shape

In [ ]:
# Evalúa los 3 algoritmos por validación cruzada temporal (media +/- std)
results, pipelines = benchmark(X, y, n_splits=5)
results

In [ ]:
# Detalle por fold del mejor modelo (para ver la estabilidad)
best_name = results.iloc[0]['model']
agg, folds = cross_validate_model(X, y, model_name=best_name, n_splits=5)
folds

In [ ]:
# El pipeline final del mejor modelo (ya ajustado con todos los datos)
best_pipeline = pipelines[best_name]
print('Mejor modelo:', best_name, '| pasos:', [n for n, _ in best_pipeline.steps])
save_model(best_pipeline)  # -> artifacts/models/model.pkl

In [ ]:
# Predicciones del mejor modelo sobre las últimas filas
preds = predict(best_pipeline, X.tail(50))
save_predictions(preds)  # -> artifacts/predictions/preds.csv
preds.head()

## LLMs

Línea de exploración complementaria: usar un LLM para enriquecer features
(p. ej. análisis de sentimiento de noticias de commodities, extracción de
eventos macro) o para explicar las predicciones del modelo.

Cargar la API key desde `.env` (no hardcodear credenciales).

In [ ]:
# Ejemplo con el SDK de OpenAI (instalado en requirements):
# import os; from openai import OpenAI
# client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
# resp = client.responses.create(model='gpt-4o-mini',
#     input='Resume el sentimiento de estas noticias de commodities...')